### IMPORTS

In [ ]:
import nltk
import pandas as pd
import numpy as np
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import unicodedataplus
from utils import scripture_guess

### Data preprocessing

In [ ]:
#Opening the csv as pandas DataFrame
df = pd.read_csv("train_submission.csv")

# We get the texts, labels and unique classes as lists
X = df['Text'].to_list()
y = df['Label'].to_list()
labels = df.groupby('Label').first().reset_index()['Label'].to_list()

# Split the dataset as train and validation datasets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.1)

# Check if every language is in the training set:
if len(set(y_test)) != len(labels):
    print(f"Il manque {-len(set(y_test)) + len(labels)} langues!")

Il manque 11 langues!


Vectorizing the texts using character N-grams

In [ ]:
# Using TF-IDF vectorizer to get the feature matrix
vectorizer = TfidfVectorizer(analyzer="char",lowercase=False,ngram_range=(1,5), max_features=100000)
X_features = vectorizer.fit_transform(X_train)

# Solving the classification problem using Naive Bayes Classifier
model = MultinomialNB()
model.fit(X_features,y_train)

MultinomialNB()

In [ ]:
# Predicting labels on validation set
y_pred = model.predict(vectorizer.transform(X_test))

# Computing accuracy
print("Accuracy: ", accuracy_score(y_test,y_pred))

Accuracy:  0.8124344176285414


### Further analysis of the results

In [ ]:
df_test = pd.DataFrame({"Text":X_test})

df_test["true_label"] = y_test  # Add true labels to the DataFrame
df_test["pred_label"] = y_pred  # Add predicted labels

writing_list = scripture_guess(X)

df_test['writing_systems'] = scripture_guess(df_test['Text'])

In [20]:
df_test.head(20)

,Text,true_label,pred_label,writing_systems
0,Betanzos munisipyu nisqaqa huk ñiqin munisipyu...,que,que,Latin
1,Comisia ar fi putut de exemplu începând cu et...,ron,ron,Latin
2,Иса Алланынъ Сарайында адамларны огреткенде ...,crh,crh,Cyrillic
3,ampy antonony basy mahenina omby sahaza A...,mlg,mlg,Latin
4,Myoyo ngasipasosekwa soni kuti jwine jwalijose...,yao,yao,Latin
5,mi tolsanji da poi mi pu pensi,jbo,djk,Latin
6,“Fondun zəmanət verəcəyi kreditlərin parametrə...,aze,aze,Latin
7,As relayed in a press statement on Monday Cab...,smo,pcm,Latin
8,Madrid bacarê İspanyaa u mıntıqa ra vartey ê n...,diq,diq,Latin
9,Berikut beberapa pertanyaan tentang ICT SMK,ind,msa,Latin


In [ ]:
# Computing accuracy per writing system

accuracy_by_writing_system = df_test.groupby("writing_systems").apply(
    lambda group: accuracy_score(group["true_label"].astype(str), group["pred_label"].astype(str))
)

# Convert to a DataFrame for readability
accuracy_df = accuracy_by_writing_system.reset_index().rename(columns={0: "accuracy"})

accuracy_df.nsmallest(n=6,columns=["accuracy"])

C:\Users\arthu\AppData\Local\Temp\ipykernel_19548\560744286.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  accuracy_by_writing_system = df_test.groupby("writing_systems").apply(


,writing_systems,accuracy
16,Inherited,0.000000
0,Arabic,0.568935
31,Tibetan,0.628571
4,Common,0.709924
25,Oriya,0.712644
12,Han,0.776062
